In [13]:
import numpy as np
from scipy.stats import norm, multivariate_normal

gird

In [6]:
def generate_x_grid(x, n_grid=1000):
    
    margin = 0.1 * (np.max(x) - np.min(x))
    x_grid = np.linspace(x_min - margin, x_max + margin, n_grid)
    
    return x_grid

### **iid data**

**Gaussian distribution**

$$ 
X_1, \ldots, X_n \overset{\text{iid}}{\sim} \mathcal{N}(\mu, \sigma^2)
$$

In [14]:
def generate_normal (n, mean=0.0, sd=1.0, seed=None):
   
    rng = np.random.default_rng(seed)
    x = rng.normal(loc=mean, scale=sd, size=n)
    
    return x

**Gaussian mixture distribution**

$$
X_1,\ldots,X_n \overset{\text{iid}}{\sim}
w\,\mathcal{N}(\mu_1,\sigma_1^2)
+
(1-w)\,\mathcal{N}(\mu_2,\sigma_2^2),
\qquad 0 \leq w \leq 1
$$

In [15]:
def generate_gaussian_mixture(n, weights=(0.5, 0.5), means=(-2.0, 2.0), sds=(0.5, 1.0), seed=None ):
    
    rng = np.random.default_rng(seed)
    components = rng.choice(len(weights), size=n, p=weights)     # 두 개(len(weight)))의 선택지 중 뽑을 확률 각각 weight 로 n개를 뽑음

    x = rng.normal(loc=means[components], scale=sds[components])

    return x

### **Dependent time series data**

**AR(1)**

$$
X_t = \phi_1 X_{t-1} + \varepsilon_t,
$$

$$
\varepsilon_t \overset{\text{iid}}{\sim} \mathcal{N}(0,\sigma^2),
$$

$$
\qquad t = 1,\ldots,n.
$$

If $|\phi_1| < 1$, a stationary distribution exists.

In [16]:
def generate_ar1(n, phi=0.7, sigma=1.0, x0=0.0, seed=None):

    rng = np.random.default_rng(seed)
    x = np.zeros(n)
    epsilon = rng.normal(loc=0.0, scale=sigma, size=n)
    
    for t in range(1, n):
        x[0] = x0
        x[t] = phi * x[t-1] + epsilon[t]

    return x

**GARCH(1,1)**

$$
X_t = \sigma_t \varepsilon_t,
$$

$$
\sigma_t^2
=
\omega
+
\alpha X_{t-1}^2
+
\beta \sigma_{t-1}^2,
$$

$$
\varepsilon_t \overset{\text{iid}}{\sim} \mathcal{N}(0,1),
$$

$$
t=1,\ldots,n.
$$

$X_t$ : return,

$\sigma_t^2$ : conditional variance,

$\epsilon_t$ : standard normal innovation.

If $\alpha + \beta < 1$, a stationary variance exists.

In [17]:
def generate_garch11(n, omega=0.1, alpha=0.1, beta=0.8, seed=None):
  
    rng = np.random.default_rng(seed)

    x = np.zeros(n)
    sigma2 = np.zeros(n)
    epsilon = rng.normal(loc=0.0, scale=1.0, size=n)

    # unconditional variance
    sigma2[0] = omega / (1 - alpha - beta)
    x[0] = np.sqrt(sigma2[0]) * epsilon[0]

    for t in range(1, n):
        sigma2[t] = omega + alpha * x[t - 1] ** 2 + beta * sigma2[t - 1]
        x[t] = np.sqrt(sigma2[t]) * epsilon[t]

    return x, sigma2

### **multivariate**


 **Bivariate VAR(1) model**



$$
\mathbf X_{t+1}=A\mathbf X_t+\boldsymbol\epsilon_{t+1}
$$

where

$$
A=\begin{pmatrix}0.6 & 0.2\\0.1 & 0.5\end{pmatrix}
$$

and

$$
\boldsymbol\epsilon_{t+1}\sim N(\mathbf 0,\Sigma)
$$

with

$$
\Sigma=\begin{pmatrix}1 & 0.6\\0.6 & 1\end{pmatrix}
$$

<br>
=>
$$
\begin{pmatrix}X_{t+1}^{(1)}\\X_{t+1}^{(2)}\end{pmatrix}
=\begin{pmatrix}0.6 & 0.2\\0.1 & 0.5\end{pmatrix}
\begin{pmatrix}X_t^{(1)}\\X_t^{(2)}\end{pmatrix}
+\begin{pmatrix}\epsilon_{t+1}^{(1)}\\\epsilon_{t+1}^{(2)}\end{pmatrix}
$$

<br>

$$
X_{t+1}^{(1)}=0.6X_t^{(1)}+0.2X_t^{(2)}+\epsilon_{t+1}^{(1)}
$$

$$
X_{t+1}^{(2)}=0.1X_t^{(1)}+0.5X_t^{(2)}+\epsilon_{t+1}^{(2)}
$$


In [2]:
def generate_var1(T=1000, burn_in=200, seed=123):

    rng = np.random.default_rng(seed)

    A = np.array([
        [0.6, 0.2],
        [0.1, 0.5] 
    ])

    Sigma = np.array([
        [1.0, 0.6],
        [0.6, 1.0]
    ])

    total_T = T + burn_in

    X = np.zeros((total_T, 2))

    eps = rng.multivariate_normal(mean=np.zeros(2), cov=Sigma, size=total_T)

    for t in range(total_T - 1):
        X[t + 1] = A @ X[t] + eps[t + 1]

    X = X[burn_in:]

    return X, A, Sigma

In [7]:
def true_var1_conditional_density(y, x_t, A, Sigma):

    y = np.asarray(y)
    x_t = np.asarray(x_t)

    mean_cond = A @ x_t

    density = multivariate_normal.pdf(y, mean=mean_cond, cov=Sigma)

    return density

### **Multivariate VAR(3) model**

**3-dimensional VAR(3) model**

The model is

$$
\mathbf{X}_{t+1} = A_1\mathbf{X}_t + A_2\mathbf{X}_{t-1} + A_3\mathbf{X}_{t-2} + \varepsilon_{t+1}.
$$

where

$$
\mathbf{X}_t = \begin{pmatrix} X_t^{(1)} \\ X_t^{(2)} \\ X_t^{(3)} \end{pmatrix}.
$$

Coefficient matrices :

$$
A_1 = \begin{pmatrix} 0.4 & 0.1 & 0.1 \\ 0.1 & 0.3 & 0.1 \\ 0.1 & 0.1 & 0.3 \end{pmatrix}, \quad A_2 = \begin{pmatrix} 0.2 & 0.1 & 0.0 \\ 0.1 & 0.2 & 0.1 \\ 0.0 & 0.1 & 0.2 \end{pmatrix}, \quad A_3 = \begin{pmatrix} 0.1 & 0.0 & 0.0 \\ 0.0 & 0.1 & 0.0 \\ 0.0 & 0.0 & 0.1 \end{pmatrix}.
$$

Error term :

$$
\varepsilon_{t+1} \sim N(0,\Sigma).
$$

with

$$
\Sigma = \begin{pmatrix} 1.0 & 0.5 & 0.3 \\ 0.5 & 1.5 & 0.4 \\ 0.3 & 0.4 & 0.7 \end{pmatrix}.
$$

Therefore,

$$
\begin{pmatrix} X_{t+1}^{(1)} \\ X_{t+1}^{(2)} \\ X_{t+1}^{(3)} \end{pmatrix} = A_1\begin{pmatrix} X_t^{(1)} \\ X_t^{(2)} \\ X_t^{(3)} \end{pmatrix} + A_2\begin{pmatrix} X_{t-1}^{(1)} \\ X_{t-1}^{(2)} \\ X_{t-1}^{(3)} \end{pmatrix} + A_3\begin{pmatrix} X_{t-2}^{(1)} \\ X_{t-2}^{(2)} \\ X_{t-2}^{(3)} \end{pmatrix} + \begin{pmatrix} \varepsilon_{t+1}^{(1)} \\ \varepsilon_{t+1}^{(2)} \\ \varepsilon_{t+1}^{(3)} \end{pmatrix}.
$$


In [4]:
def generate_var3_3d(T=1000, burn_in=300):

    
    rng = np.random.default_rng(seed=123)

    A1 = np.array([
        [0.4, 0.1, 0.1],
        [0.1, 0.3, 0.1],
        [0.1, 0.1, 0.3]
    ])

    A2 = np.array([
        [0.2, 0.1, 0.0],
        [0.1, 0.2, 0.1],
        [0.0, 0.1, 0.2]
    ])

    A3 = np.array([
        [0.1, 0.0, 0.0],
        [0.0, 0.1, 0.0],
        [0.0, 0.0, 0.1]
    ])
    Sigma = np.array([
        [1.0, 0.5, 0.3],
        [0.5, 1.5, 0.4],
        [0.3, 0.4, 0.7]
    ])

    total_T = T + burn_in

    X = np.zeros((total_T, 3))

    eps = rng.multivariate_normal(mean=np.zeros(3), cov=Sigma, size=total_T)

    for t in range(2, total_T - 1):

        X[t + 1] = ( A1 @ X[t] + A2 @ X[t - 1] + A3 @ X[t - 2] + eps[t + 1] )

    X = X[burn_in:]

    return X, A1, A2, A3, Sigma

In [2]:
def true_var3_3d_conditional_density(y, x_t, x_t_minus_1, x_t_minus_2, A1, A2, A3, Sigma):

    y = np.asarray(y)
    x_t = np.asarray(x_t)
    x_t_minus_1 = np.asarray(x_t_minus_1)
    x_t_minus_2 = np.asarray(x_t_minus_2)

    mean_cond = (A1 @ x_t + A2 @ x_t_minus_1 + A3 @ x_t_minus_2)

    density = multivariate_normal.pdf(y, mean=mean_cond, cov=Sigma)

    return density